In [ ]:
import os
import json
import logging
import numpy as np
import pandas as pd
from datetime  import datetime
from  sklearn.datasets  import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import  StandardScaler
from sklearn.metrics import  accuracy_score,f1_score,roc_auc_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

data = load_breast_cancer()
X ,y = data.data, data.target
feature_names = list(data.feature_names)

X_train,X_test ,  y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print("Dataset loaded: Breast Cancer")
print(f"  Train: {X_train.shape}  Test: {X_test.shape}")
print(f"  Features: {len(feature_names)}")
print(f"Classes :{np.bincount(y_train) } (0=malignant, 1=benign)")

Dataset loaded: Breast Cancer
  Train: (455, 30)  Test: (114, 30)
  Features: 30
Classes :[170 285] (0=malignant, 1=benign)


In [ ]:
# ============================================================
# CI/CD for ML — what it automates
#
# Traditional CI/CD:  code changes → test → build → deploy
# ML CI/CD:           code OR data OR model changes
#                     → test → train → evaluate → deploy
#
# Three triggers in ML CI/CD:
#   1. Code trigger:  pipeline code changed (git push)
#   2. Data trigger:  new data arrived, schema changed
#   3. Model trigger: model performance dropped below threshold
# ============================================================

print("CI/CD for ML — trigger types:\n")

CI_CD_TRIGGERS = {
    "Code trigger": {
        "when":   "Developer pushes changes to training pipeline or model code",
        "action": "Run unit tests → retrain on full data → evaluate → deploy if improved",
        "github_actions": "on: push to main branch",
    },
    "Data trigger": {
        "when":   "New batch of data arrives (scheduled or event-driven)",
        "action": "Validate schema → run data quality checks → retrain → evaluate → deploy",
        "github_actions": "on: schedule (cron) or repository_dispatch event",
    },
    "Model trigger": {
        "when":   "Monitoring detects accuracy drop or data drift above threshold",
        "action": "Alert → auto-retrain or page on-call engineer → evaluate → deploy",
        "github_actions": "on: repository_dispatch from monitoring system",
    },
}

for trigger, details in CI_CD_TRIGGERS.items():
    print(f"{trigger}:")
    for k, v in details.items():
        print(f"  {k:<18} {v}")
    print()

CI/CD for ML — trigger types:

Code trigger:
  when               Developer pushes changes to training pipeline or model code
  action             Run unit tests → retrain on full data → evaluate → deploy if improved
  github_actions     on: push to main branch

Data trigger:
  when               New batch of data arrives (scheduled or event-driven)
  action             Validate schema → run data quality checks → retrain → evaluate → deploy
  github_actions     on: schedule (cron) or repository_dispatch event

Model trigger:
  when               Monitoring detects accuracy drop or data drift above threshold
  action             Alert → auto-retrain or page on-call engineer → evaluate → deploy
  github_actions     on: repository_dispatch from monitoring system



In [ ]:
# ============================================================
# GitHub Actions workflow — what a real ML CI/CD YAML looks like
# We print it here; in production this lives at .github/workflows/ml_pipeline.yml
# ============================================================

ML_GITHUB_ACTIONS_YAML = """
name: ML Training Pipeline

on:
  push:
    branches: [main]
    paths:
      - 'src/train.py'       # only trigger when training code changes
      - 'data/new_data.csv'  # or when new data is committed
  schedule:
    - cron: '0 2 * * 1'     # every Monday at 2am — weekly retrain

jobs:
  ml-pipeline:
    runs-on: ubuntu-latest
    steps:

      - name: Checkout code
        uses: actions/checkout@v3

      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run data validation tests
        run: pytest tests/test_data_quality.py

      - name: Run unit tests on pipeline
        run: pytest tests/test_pipeline.py

      - name: Train model
        run: python src/train.py --config config.yaml

      - name: Evaluate model
        run: python src/evaluate.py --threshold 0.90

      - name: Register model if improved
        run: python src/register_model.py --promote-if-better

      - name: Deploy to staging
        if: success()
        run: python src/deploy.py --env staging
"""

print("GitHub Actions workflow (.github/workflows/ml_pipeline.yml):")
print(ML_GITHUB_ACTIONS_YAML)

GitHub Actions workflow (.github/workflows/ml_pipeline.yml):

name: ML Training Pipeline

on:
  push:
    branches: [main]
    paths:
      - 'src/train.py'       # only trigger when training code changes
      - 'data/new_data.csv'  # or when new data is committed
  schedule:
    - cron: '0 2 * * 1'     # every Monday at 2am — weekly retrain

jobs:
  ml-pipeline:
    runs-on: ubuntu-latest
    steps:

      - name: Checkout code
        uses: actions/checkout@v3

      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run data validation tests
        run: pytest tests/test_data_quality.py

      - name: Run unit tests on pipeline
        run: pytest tests/test_pipeline.py

      - name: Train model
        run: python src/train.py --config config.yaml

      - name: Evaluate model
        run: python src/evaluate.py --threshold 0.90


In [ ]:
def test_data_quality(X: np.ndarray, y: np.ndarray,
                      feature_names: list) -> dict:
    """Data quality checks that run before any training."""
    results = {}

    # Check 1: No missing values
    results["no_missing_values"] = not pd.DataFrame(X, columns=feature_names).isnull().any().any()

    # Check 2: Expected number of features
    results["correct_feature_count"] = X.shape[1] == 30

    # Check 3: Target is binary
    results["binary_target"] = len(np.unique(y)) == 2

    # Check 4: Class balance not severely skewed (no class < 10%)
    class_ratios = np.bincount(y) / len(y)
    results["class_balance_ok"] = all(r > 0.10 for r in class_ratios)

    # Check 5: No constant features (zero variance)
    variances = np.var(X, axis=0)
    results["no_constant_features"] = all(variances > 0)

    return results

In [ ]:
def test_model_pipeline(pipeline: Pipeline,
                        X_test: np.ndarray,
                        y_test: np.ndarray,
                        min_accuracy: float = 0.90) -> dict:
    """Model quality gates that must pass before deployment."""
    results = {}

    preds     = pipeline.predict(X_test)
    proba     = pipeline.predict_proba(X_test)[:, 1]

    results["accuracy_above_threshold"] = accuracy_score(y_test, preds) >= min_accuracy
    results["f1_above_0.90"]            = f1_score(y_test, preds) >= 0.90
    results["roc_auc_above_0.95"]       = roc_auc_score(y_test, proba) >= 0.95
    results["no_nan_predictions"]       = not np.isnan(preds).any()
    results["output_is_binary"]         = set(np.unique(preds)).issubset({0, 1})

    return results

In [ ]:
def run_ci_gate(pipeline, X_train, y_train, X_test, y_test, feature_names, label=""):
    """Runs the full CI gate and prints a pass/fail summary."""
    print(f"=== Data quality tests {label} ===")
    data_results = test_data_quality(X_train, y_train, feature_names)
    for test, passed in data_results.items():
        print(f"  {'✅' if passed else '❌'} {test}")

    print(f"\n=== Model quality gate tests {label} ===")
    model_results = test_model_pipeline(pipeline, X_test, y_test)
    for test, passed in model_results.items():
        print(f"  {'✅' if passed else '❌'} {test}")

    all_passed = all(data_results.values()) and all(model_results.values())
    print(f"\nCI gate: {'✅ PASSED — proceed to deploy' if all_passed else '❌ FAILED — block deployment'}")
    return all_passed

In [ ]:
# --- Scenario 1: a properly trained model — expect the gate to pass ---
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  RandomForestClassifier(n_estimators=100, random_state=42))
])
pipeline.fit(X_train, y_train)

run_ci_gate(pipeline, X_train, y_train, X_test, y_test, feature_names, label="(healthy model)")

=== Data quality tests (healthy model) ===
  ✅ no_missing_values
  ✅ correct_feature_count
  ✅ binary_target
  ✅ class_balance_ok
  ✅ no_constant_features

=== Model quality gate tests (healthy model) ===
  ✅ accuracy_above_threshold
  ✅ f1_above_0.90
  ✅ roc_auc_above_0.95
  ✅ no_nan_predictions
  ✅ output_is_binary

CI gate: ✅ PASSED — proceed to deploy


True

In [ ]:
# --- Scenario 2: a deliberately weakened model — expect the gate to fail ---
# This simulates what CI catches: a bad training run should never reach production.
weak_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  RandomForestClassifier(n_estimators=1, max_depth=1, random_state=42))
])
weak_pipeline.fit(X_train, y_train)

print()
run_ci_gate(weak_pipeline, X_train, y_train, X_test, y_test, feature_names, label="(deliberately weak model)")


=== Data quality tests (deliberately weak model) ===
  ✅ no_missing_values
  ✅ correct_feature_count
  ✅ binary_target
  ✅ class_balance_ok
  ✅ no_constant_features

=== Model quality gate tests (deliberately weak model) ===
  ✅ accuracy_above_threshold
  ✅ f1_above_0.90
  ❌ roc_auc_above_0.95
  ✅ no_nan_predictions
  ✅ output_is_binary

CI gate: ❌ FAILED — block deployment


False

In [ ]:
import json
from dataclasses import dataclass, field, asdict
from typing import Optional

In [ ]:
@dataclass
class ModelVersion:
  version: int
  name:str
  stage:str
  accuracy:float
  f1_score:float
  roc_auc:float
  parameters:dict
  trained_at:str
  promoted_at:Optional[str]=None
  notes : str =""

  def to_dict(self):
    return asdict(self)



In [ ]:
class ModelRegistry:
  """Lightweight  Local model  registry - mirrors MLFlow model registry concept"""

  def __init__(self):
    self.versions: list[ModelVersion]= []
    self._next_version =1

  def register(self,name:str,metrics:dict,parameters:dict,notes: str = "") -> ModelVersion:
    v = ModelVersion(
        version = self._next_version,
        name = name,
        stage = "staging",
        accuracy= metrics["accuracy"],
        f1_score =metrics["F1"],
        roc_auc = metrics["roc_auc"],
        parameters = parameters,
        trained_at  = datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        notes = notes
    )

    self.versions.append(v)
    self._next_version +=1
    return v

  def promote_to_production(self,version:int):
    """Demonstrate current  producation  -> archieved, promote staging -> producation"""
    for v in self.versions:
      if v.stage == "production":
        v.stage = "archieved"
    for v in self.versions:
      if v.version == version:
        v.stage = "production"
        v.promoted_at = datetime.now().strftime(("%Y-%m-%d %H:%M:%S"))

  def  best_staging_version(self,metrics:str='accuracy') -> Optional[ModelVersion]:
     """Selects the staging version with the highest value of `metric`."""
     staged= [v for v in  self.versions  if v.stage == "staging"]
     if not staged:
        return None
     return  max(staged,key=lambda v: getattr(v,metrics))


  def  get_production(self) -> Optional[ModelVersion]:
    for v in self.versions:
      if v.stage == "production":
         return v
    return  None


  def leaderboard(self):
        print(f"\n{'Ver':>4} {'Name':<25} {'Stage':<12} {'Acc':>7} {'F1':>7} {'AUC':>7}  Trained")
        print("-" * 80)
        for v in sorted(self.versions, key=lambda x: x.version, reverse=True):
            print(f"{v.version:>4} {v.name:<25} {v.stage:<12} "
                  f"{v.accuracy:>7.3f} {v.f1_score:>7.3f} {v.roc_auc:>7.3f}  {v.trained_at}")

In [ ]:
registry = ModelRegistry()

In [ ]:
def evaluate_pipeline(pipeline,X_test, y_test) -> dict:
  preds = pipeline.predict(X_test)
  proba = pipeline.predict_proba(X_test) [:,1]
  return {
      'accuracy':round(accuracy_score(y_test,preds),4),
      'F1':  round(f1_score(y_test,preds),4),
      "roc_auc":round(roc_auc_score(y_test,proba),4),
  }

In [ ]:
# Version 1: Logistic Regression baseline
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(C=1.0, max_iter=500))])
lr_pipe.fit(X_train, y_train)
v1 = registry.register("LogisticRegression-v1", evaluate_pipeline(lr_pipe, X_test, y_test),
                         {"C": 1.0, "max_iter": 500})

In [ ]:
rf_pipe = Pipeline([("scaler",StandardScaler()), ("model",RandomForestClassifier(n_estimators=50,random_state=42))])
rf_pipe.fit(X_train,y_train)
v2 = registry.register("RandomForest-v2", evaluate_pipeline(rf_pipe, X_test, y_test),
                         {"n_estimators": 50})

In [ ]:
# Version 3: Random Forest tuned
rf_pipe2 = Pipeline([("scaler", StandardScaler()), ("model", RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42))])
rf_pipe2.fit(X_train, y_train)
v3 = registry.register("RandomForest-v3-tuned", evaluate_pipeline(rf_pipe2, X_test, y_test),
                         {"n_estimators": 200, "max_depth": 10},
                         notes="Tuned depth")

In [ ]:
best = registry.best_staging_version(metrics="accuracy")
registry.promote_to_production(version=best.version)

print("=== Model Registry===")
registry.leaderboard()
print(f"\n current  producation  models:{registry.get_production().name}")


=== Model Registry===

 Ver Name                      Stage            Acc      F1     AUC  Trained
--------------------------------------------------------------------------------
   3 RandomForest-v3-tuned     staging        0.956   0.966   0.993  2026-08-01 10:58:02
   2 RandomForest-v2           staging        0.956   0.966   0.992  2026-08-01 10:58:01
   1 LogisticRegression-v1     production     0.983   0.986   0.995  2026-08-01 10:58:01

 current  producation  models:LogisticRegression-v1


In [ ]:
#{"timestamp": "...", "user_id": 42, "prediction": 1,
#                "confidence": 0.94, "model_version": 3, "latency_ms": 12}


class MLLogger:
    """Structured JSON logger for ML prediction events."""

    def __init__(self, model_version: int, log_file: str = "ml_predictions.jsonl"):
        self.model_version = model_version
        self.log_file      = log_file
        self.records       = []

    def log_prediction(self, input_features: np.ndarray,
                       prediction: int, confidence: float,
                       latency_ms: float, user_id: str = "anonymous"):
        record = {
            "timestamp":     datetime.now().isoformat(),
            "user_id":       user_id,
            "model_version": self.model_version,
            "prediction":    int(prediction),
            "confidence":    round(float(confidence), 4),
            "latency_ms":    round(latency_ms, 2),
            "input_mean":    round(float(np.mean(input_features)), 4),
            "input_std":     round(float(np.std(input_features)), 4),
        }
        self.records.append(record)
        return record

    def log_batch_metrics(self, accuracy: float, f1: float,
                          n_samples: int, period: str):
        record = {
            "timestamp":     datetime.now().isoformat(),
            "event_type":    "batch_evaluation",
            "model_version": self.model_version,
            "period":        period,
            "accuracy":      round(accuracy, 4),
            "f1_score":      round(f1, 4),
            "n_samples":     n_samples,
        }
        self.records.append(record)
        return record

    def summary(self):
        preds    = [r for r in self.records if "prediction" in r]
        metrics  = [r for r in self.records if r.get("event_type") == "batch_evaluation"]
        print(f"Total prediction logs: {len(preds)}")
        print(f"Total metric logs:     {len(metrics)}")
        if preds:
            confs = [r["confidence"] for r in preds]
            lats  = [r["latency_ms"] for r in preds]
            print(f"Avg confidence:        {np.mean(confs):.3f}")
            print(f"Avg latency (ms):      {np.mean(lats):.2f}")
            print(f"Low confidence (<0.7): {sum(1 for c in confs if c < 0.7)}")

In [ ]:
import time
production_pipeline = {1: lr_pipe, 2: rf_pipe, 3: rf_pipe2}[registry.get_production().version]

logger = MLLogger(model_version=registry.get_production().version)


# Simulate prediction logging
print("=== Simulating 20 prediction requests ===\n")
for i in range(20):
    idx        = np.random.randint(len(X_test))
    start      = time.time()
    pred       = production_pipeline.predict(X_test[idx:idx+1])[0]
    conf       = production_pipeline.predict_proba(X_test[idx:idx+1])[0].max()
    latency    = (time.time() - start) * 1000 + np.random.uniform(2, 15)
    record     = logger.log_prediction(X_test[idx], pred, conf, latency, user_id=f"user_{i}")
    print(f"  [{record['timestamp'][11:19]}] user_{i:<3} → pred={pred} conf={conf:.3f} lat={latency:.1f}ms")

print()
logger.summary()

# Log a batch evaluation
metrics = evaluate_pipeline(production_pipeline, X_test, y_test)
logger.log_batch_metrics(metrics["accuracy"], metrics["F1"],
                          len(X_test), period="2026-07")

print(f"\nBatch evaluation logged: acc={metrics['accuracy']} f1={metrics['F1']}")

=== Simulating 20 prediction requests ===

  [10:58:02] user_0   → pred=0 conf=0.968 lat=14.3ms
  [10:58:02] user_1   → pred=0 conf=1.000 lat=12.7ms
  [10:58:02] user_2   → pred=1 conf=0.998 lat=10.9ms
  [10:58:02] user_3   → pred=0 conf=1.000 lat=4.4ms
  [10:58:02] user_4   → pred=0 conf=1.000 lat=14.3ms
  [10:58:02] user_5   → pred=1 conf=0.992 lat=4.9ms
  [10:58:02] user_6   → pred=0 conf=0.994 lat=3.3ms
  [10:58:02] user_7   → pred=1 conf=1.000 lat=12.4ms
  [10:58:02] user_8   → pred=1 conf=0.999 lat=5.8ms
  [10:58:02] user_9   → pred=1 conf=0.973 lat=16.1ms
  [10:58:02] user_10  → pred=1 conf=0.984 lat=7.1ms
  [10:58:02] user_11  → pred=0 conf=0.948 lat=3.2ms
  [10:58:02] user_12  → pred=1 conf=0.902 lat=6.9ms
  [10:58:02] user_13  → pred=0 conf=1.000 lat=8.3ms
  [10:58:02] user_14  → pred=1 conf=0.975 lat=15.7ms
  [10:58:02] user_15  → pred=0 conf=1.000 lat=9.0ms
  [10:58:02] user_16  → pred=1 conf=1.000 lat=11.1ms
  [10:58:02] user_17  → pred=0 conf=1.000 lat=9.8ms
  [10:58:02] 

In [ ]:
from scipy import stats

def ks_drift_test(reference: np.ndarray, current: np.ndarray,
                  feature_names: list, alpha: float = 0.05) -> pd.DataFrame:
    """
    Kolmogorov-Smirnov test per feature.
    p-value < alpha → distributions are significantly different → drift detected.
    """
    results = []
    for i, name in enumerate(feature_names):
        stat, p_val = stats.ks_2samp(reference[:, i], current[:, i])
        results.append({
            "feature":    name,
            "ks_stat":    round(stat, 4),
            "p_value":    round(p_val, 4),
            "drift":      p_val < alpha,
        })
    return pd.DataFrame(results)

In [ ]:
def psi(reference: np.ndarray, current: np.ndarray, bins: int = 5) -> float:
    """
    Population Stability Index for a single feature.
    PSI < 0.1:  no drift
    PSI 0.1-0.2: moderate drift — monitor
    PSI > 0.2:   significant drift — investigate / retrain

    Bin edges are extended to +/-inf so current-data points outside the
    reference range are still counted (np.histogram would otherwise drop them).

    bins=5 (not the textbook default of 10) because our reference/current
    sets here are small (~450 / ~114 rows). With more bins than the sample
    size supports, tail bins can land on zero points by chance alone, which
    inflates PSI even when there's no real distribution shift. In production,
    with thousands of rows per batch, 10+ bins is standard.
    """
    edges = np.histogram_bin_edges(reference, bins=bins)
    edges = edges.copy()
    edges[0], edges[-1] = -np.inf, np.inf

    ref_hist, _ = np.histogram(reference, bins=edges)
    cur_hist, _ = np.histogram(current, bins=edges)

    ref_pct = (ref_hist + 1e-6) / (ref_hist.sum() + 1e-6 * bins)
    cur_pct = (cur_hist + 1e-6) / (cur_hist.sum() + 1e-6 * bins)

    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))

In [ ]:
X_clean = X_test.copy()
X_drifted= X_test.copy()
X_drifted[:,:5] = X_drifted[:,:5 ] * 1.8 + 3.0


In [ ]:
print("=== KS drift test: clean data vs training ===")
ks_clean = ks_drift_test(X_train,X_clean,feature_names)
drifted_features = ks_clean['drift']
print(f"features with drift detected: {len(drifted_features)} / {len(feature_names)}")
print(ks_clean.head(6).to_string(index=False))

=== KS drift test: clean data vs training ===
features with drift detected: 30 / 30
         feature  ks_stat  p_value  drift
     mean radius   0.0692   0.7433  False
    mean texture   0.0760   0.6342  False
  mean perimeter   0.0821   0.5388  False
       mean area   0.0670   0.7774  False
 mean smoothness   0.0763   0.6264  False
mean compactness   0.0689   0.7496  False


In [ ]:
print("\n=== KS drift test: drifted data vs training ===")
ks_drifted = ks_drift_test(X_train, X_drifted, feature_names)
drifted_features = ks_drifted[ks_drifted["drift"]]
print(f"Features with drift detected: {len(drifted_features)} / {len(feature_names)}")
print(ks_drifted.head(8).to_string(index=False))


=== KS drift test: drifted data vs training ===
Features with drift detected: 5 / 30
            feature  ks_stat  p_value  drift
        mean radius   0.8880   0.0000   True
       mean texture   0.9013   0.0000   True
     mean perimeter   0.7892   0.0000   True
          mean area   0.5432   0.0000   True
    mean smoothness   1.0000   0.0000   True
   mean compactness   0.0689   0.7496  False
     mean concavity   0.1214   0.1250  False
mean concave points   0.1131   0.1775  False


In [ ]:
# PSI on first feature
psi_clean   = psi(X_train[:, 0], X_clean[:, 0])
psi_drifted = psi(X_train[:, 0], X_drifted[:, 0])
print(f"\nPSI — mean radius (clean):   {psi_clean:.4f}  {'✅ stable' if psi_clean < 0.1 else '⚠️ drift'}")
print(f"PSI — mean radius (drifted): {psi_drifted:.4f}  {'✅ stable' if psi_drifted < 0.1 else '⚠️ significant drift — retrain'}")


PSI — mean radius (clean):   0.0041  ✅ stable
PSI — mean radius (drifted): 16.6174  ⚠️ significant drift — retrain


In [ ]:
@dataclass
class Alert:
    severity:  str    # critical / warning / info
    metric:    str
    value:     float
    threshold: float
    message:   str
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())

In [ ]:
class AlertManager:
    def __init__(self):
        self.alerts: list[Alert] = []
        self.rules = []

    def add_rule(self, metric: str, threshold: float,
                 severity: str, condition: str = "below"):
        self.rules.append({
            "metric": metric, "threshold": threshold,
            "severity": severity, "condition": condition
        })

    def evaluate(self, metrics: dict):
        for rule in self.rules:
            val  = metrics.get(rule["metric"])
            if val is None: continue
            if rule["condition"] == "below"  and val < rule["threshold"]:
                self._fire(rule, val)
            elif rule["condition"] == "above" and val > rule["threshold"]:
                self._fire(rule, val)

    def _fire(self, rule: dict, val: float):
        alert = Alert(
            severity  = rule["severity"],
            metric    = rule["metric"],
            value     = val,
            threshold = rule["threshold"],
            message   = (f"[{rule['severity'].upper()}] {rule['metric']} = {val:.4f} "
                         f"({'below' if rule['condition'] == 'below' else 'above'} "
                         f"threshold {rule['threshold']})")
        )
        self.alerts.append(alert)
        print(f"  🚨 {alert.message}")

    def summary(self):
        if not self.alerts:
            print("  ✅ No alerts fired.")
        else:
            by_severity = {}
            for a in self.alerts:
                by_severity.setdefault(a.severity, []).append(a)
            for sev, alerts in by_severity.items():
                print(f"  {sev.upper()}: {len(alerts)} alert(s)")



In [ ]:
alert_mgr = AlertManager()
alert_mgr.add_rule("accuracy",       threshold=0.92, severity="critical", condition="below")
alert_mgr.add_rule("f1_score",       threshold=0.90, severity="warning",  condition="below")
alert_mgr.add_rule("psi_mean_radius",threshold=0.20, severity="critical", condition="above")
alert_mgr.add_rule("avg_latency_ms", threshold=50.0, severity="warning",  condition="above")

In [ ]:
print("=== Alert evaluation: healthy production metrics ===\n")
production_metrics = {
    "accuracy":        metrics["accuracy"],
    "f1_score":        metrics["f1"],
    "psi_mean_radius": psi_clean,     # clean data — should NOT trigger drift alert
    "avg_latency_ms":  12.4,
}

=== Alert evaluation: healthy production metrics ===



KeyError: 'f1'

In [ ]:
print("=== Alert evaluation: healthy production metrics ===\n")
production_metrics = {
    "accuracy":        metrics["accuracy"],
    "f1_score":        metrics["F1"],
    "psi_mean_radius": psi_clean,     # clean data — should NOT trigger drift alert
    "avg_latency_ms":  12.4,
}

=== Alert evaluation: healthy production metrics ===



In [ ]:
alert_mgr.evaluate(production_metrics)
alert_mgr.summary()

  ✅ No alerts fired.


In [ ]:
print("\n=== Alert evaluation: degraded model scenario ===\n")
degraded_metrics = {
    "accuracy":        0.88,
    "f1_score":        0.85,
    "psi_mean_radius": psi_drifted,   # drifted data — should trigger drift alert
    "avg_latency_ms":  65.0,
}


=== Alert evaluation: degraded model scenario ===



In [ ]:
alert_mgr_deg = AlertManager()
for rule in alert_mgr.rules:
    alert_mgr_deg.add_rule(rule["metric"], rule["threshold"],
                            rule["severity"], rule["condition"])
alert_mgr_deg.evaluate(degraded_metrics)
alert_mgr_deg.summary()

  🚨 [CRITICAL] accuracy = 0.8800 (below threshold 0.92)
  🚨 [WARNING] f1_score = 0.8500 (below threshold 0.9)
  🚨 [CRITICAL] psi_mean_radius = 16.6174 (above threshold 0.2)
  🚨 [WARNING] avg_latency_ms = 65.0000 (above threshold 50.0)
  CRITICAL: 2 alert(s)
